This script filters each Blast2GO file, selecting only the rows where the 'Tags' column contains the label 'Under'.  
Then, the resulting DataFrame is sorted by the 'P-Value' column in ascending order.  
The top 5 rows from each Blast2GO file are selected and combined into a single output file.  
If a file has fewer than 5 rows, all available rows (1, 2, 3, or 4) will be included.

In [4]:
import os
import pandas as pd
import numpy as np

file = pd.read_csv("blast2go_table_bzip2.txt", sep="\t")
file = file[file["Tags"] =="[OVER]"]
file=file.sort_values(by=["P-Value"])
file = file[0:5]
file

**INITIAL FILTERING TO OBTAIN THE TOP 2 ROWS**

In [10]:
# Definir el directorio donde se encuentran los archivos blast2go
directory = "/home/alumno30/TFM/DAP/Picos/GO/"

# Obtener la lista de archivos en el directorio que contienen "blast2go" en su nombre
blast2go_files =[f for f in os.listdir(directory) if "blast2go_table" in f]

# Lista para almacenar los dataframes filtrados de cada archivo
filtered_dfs =[]


# Iterar sobre cada archivo blast2go
for filename in blast2go_files:
    file_path= os.path.join(directory, filename)
    
    #leer el archivo
    df = pd.read_csv(file_path, sep="\t")
    
    #filtrar las filas donde la columna Tags tiene "OVER"
    filtered_df = df[df["Tags"]== "[OVER]"]
    
    #Ordenar por la columna "P-Value"
    filtered_df = filtered_df.sort_values(by="P-Value")
    
    # Selecccionar las 2 primeras filas 
    top5_df =filtered_df.head(2)
    
    #agregar al dataframe general
    filtered_dfs.append(top5_df)
    
#Contactenar todos los datframes filtrados en uno solo
final_df = pd.concat(filtered_dfs, ignore_index=True)

# Eliminar duplicados en la columna 'Annotation'
final_df = final_df.drop_duplicates(subset=['GO ID'])
final_df = final_df["GO ID"]

# Definir el nombre dela rchivo de salida
output_filename = os.path.join(directory, "blast2go_top2_combined.txt")

# Guardar el dataframe final en un archivo
final_df.to_csv(output_filename, sep="\t", index=False)

print(f"Archivo combinado guardado: {output_filename}")

Archivo combinado guardado: /home/alumno30/TFM/DAP/Picos/GO/blast2go_top2_combined.txt


**COMBINING FILTERED DATA WITH ORIGINAL BLAST2GO FILES**

The previously filtered file will be merged with each of the original Blast2GO files.  
The goal is to retain the rows from the filtered file while checking if they have a match  in each corresponding Blast2GO file. Then, we will extract the "Nr Test" and "Nr Reference" columns from each Blast2GO file and save them separately.

**EXTRACTING P-VALUES OF COMMON GO IDs FOR EACH TRANSCRIPTION FACTOR**

In [11]:
# Definir el directorio donde se encuentran los archivos blast2go
directory = "/home/alumno30/TFM/DAP/Picos/GO/"  # Cambia esto por el directorio real

# Leer el archivo combinado
combined_file_path = os.path.join(directory, "blast2go_top2_combined.txt")
combined_df = pd.read_csv(combined_file_path, sep="\t")

# Obtener la lista de archivos en el directorio que contienen "blast2go" en su nombre y no son el archivo combinado
blast2go_files = [f for f in os.listdir(directory) if "blast2go" in f and f != "blast2go_top5_combined.txt"]

# Iterar sobre cada archivo blast2go
for filename in blast2go_files:
    file_path = os.path.join(directory, filename)
    
    # Leer el archivo original de blast2go
    blast2go_df = pd.read_csv(file_path, sep="\t")
    
    # Hacer el merge entre el archivo combinado y el archivo blast2go por la columna 'GO ID'
    merged_df = pd.merge(combined_df, blast2go_df[['GO ID', 'P-Value']], on='GO ID', how='left', suffixes=('', f'_{filename.replace(".txt", "")}'))
    
    # Imprimir los nombres de las columnas disponibles para verificar
    print(f"Columnas en {filename}: {merged_df.columns.tolist()}")
    
    # Definir el nombre del archivo de salida
    output_filename = os.path.join(directory, f"{filename.replace('.txt', '_result.txt')}")
    
    # Guardar el dataframe con las columnas relevantes en un archivo
    merged_df.to_csv(output_filename, sep="\t", index=False)
    
    print(f"Archivo resultado guardado: {output_filename}")


Columnas en blast2go_table_bZip5.txt: ['GO ID', 'P-Value']
Archivo resultado guardado: /home/alumno30/TFM/DAP/Picos/GO/blast2go_table_bZip5_result.txt
Columnas en blast2go_table_bZip3.txt: ['GO ID', 'P-Value']
Archivo resultado guardado: /home/alumno30/TFM/DAP/Picos/GO/blast2go_table_bZip3_result.txt
Columnas en blast2go_table_bZip6.txt: ['GO ID', 'P-Value']
Archivo resultado guardado: /home/alumno30/TFM/DAP/Picos/GO/blast2go_table_bZip6_result.txt
Columnas en blast2go_table_FK1.txt: ['GO ID', 'P-Value']
Archivo resultado guardado: /home/alumno30/TFM/DAP/Picos/GO/blast2go_table_FK1_result.txt
Columnas en blast2go_table_CopF3.txt: ['GO ID', 'P-Value']
Archivo resultado guardado: /home/alumno30/TFM/DAP/Picos/GO/blast2go_table_CopF3_result.txt
Columnas en blast2go_table_CP2.txt: ['GO ID', 'P-Value']
Archivo resultado guardado: /home/alumno30/TFM/DAP/Picos/GO/blast2go_table_CP2_result.txt
Columnas en blast2go_table_bZip7.txt: ['GO ID', 'P-Value']
Archivo resultado guardado: /home/alumno30/

KeyError: "['P-Value'] not in index"

**GENERATING THE P-VALUE MATRIX FOR TRANSCRIPTION FACTORS**

In [12]:
# Definir el directorio donde se encuentran los archivos de resultados
directory = "/home/alumno30/TFM/DAP/Picos/GO/"  # Cambia esto por el directorio real

# Obtener la lista de archivos en el directorio que contienen "_result.txt" en su nombre
result_files = [f for f in os.listdir(directory) if "_result.txt" in f]

# Inicializar un diccionario para almacenar los datos de log2
pvalue_data = {}

# Iterar sobre cada archivo de resultados
for filename in result_files:
    file_path = os.path.join(directory, filename)
    
    # Leer el archivo de resultados
    result_df = pd.read_csv(file_path, sep="\t")
    
    # Extraer el nombre de la columna para el archivo
    col_name = filename.split('_')[2]  # Suponiendo que el nombre del archivo es como 'blast2go_table_bzip2_result.txt'
    
    # Almacenar la columna 'log2' en el diccionario con el nombre extraído
    pvalue_data[col_name] = result_df.set_index('GO ID')['P-Value']

# Convertir el diccionario en un DataFrame
pvalue_matrix = pd.DataFrame(pvalue_data)

# Añadir la columna 'Annotation' al DataFrame
pvalue_matrix.insert(0, 'GO ID', pvalue_matrix.index)

# Guardar el DataFrame resultante en un archivo
output_matrix_file = os.path.join(directory, "pvalue_summary_matrix.txt")
pvalue_matrix.to_csv(output_matrix_file, sep="\t", index=False)

print(f"Matriz de pvalue guardada en: {output_matrix_file}")


Matriz de pvalue guardada en: /home/alumno30/TFM/DAP/Picos/GO/pvalue_summary_matrix.txt
